# BioRAG-X — 09 Agentic Retrieval

## Research framing
This notebook turns the retrieval stack into a bounded, measurable decision process. The agent is **not** a free-form autonomous agent: it analyzes the question, chooses a minimum retrieval plan, executes deterministic retrieval tools, assesses evidence sufficiency, and may recover once (or twice in later ablations) using query rewriting, HyDE, Query2Doc, decomposition, GraphRAG, or PageIndex. The scientific goal is to maximize evidence quality per unit latency/cost.

### Research questions
1. Does routing improve retrieval quality over a fixed hybrid baseline?
2. When does recovery help, and when does it introduce drift?
3. Does a bounded agent reduce unnecessary retrieval?
4. Which query types benefit from graph, PageIndex, expansion, HyDE, or decomposition?
5. What is the quality/latency trade-off as the retrieval budget changes?

Primary comparison: **Fixed Hybrid → Static Multi-tool → Adaptive Agentic Retrieval**.

In [ ]:
from pathlib import Path
import json, math, time, re, statistics
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import numpy as np

ROOT = Path("/mnt/data")
ARTIFACT_DIR = ROOT / "biorag_x_artifacts" / "09_agentic_retrieval"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Artifact directory:", ARTIFACT_DIR)


## 1. Load Notebook 07/08 artifacts and preserve a clean contract

In [ ]:
def first_existing(paths):
    for p in paths:
        if p.exists(): return p
    return None

candidates = [
    ROOT / "biorag_x_artifacts",
    ROOT / "BioRAG-X_artifacts",
]
for c in candidates:
    if c.exists(): print("Found artifact root:", c)

# Preferred inputs; the notebook remains executable with synthetic smoke data if prior artifacts are absent.
manifest_candidates = list(ROOT.rglob("*07*manifest*.json")) + list(ROOT.rglob("*08*manifest*.json"))
print("Candidate manifests:", [str(p) for p in manifest_candidates[:8]])


### Minimal benchmark adapter
The orchestrator consumes tool interfaces rather than hard-coding implementation details. In the full pipeline, these adapters wrap Notebook 07/08 retrievers.

In [ ]:
@dataclass
class Evidence:
    passage_id: str
    text: str
    source: str
    score: float
    rank: int = 0
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class RetrievalResult:
    tool: str
    query: str
    evidence: List[Evidence]
    latency_ms: float
    metadata: Dict[str, Any] = field(default_factory=dict)

class RetrieverAdapter:
    def __init__(self, name, fn=None):
        self.name=name; self.fn=fn
    def retrieve(self, query, top_k=10, **kwargs):
        t=time.perf_counter()
        if self.fn is None:
            out=[]
        else:
            out=self.fn(query, top_k=top_k, **kwargs)
        return RetrievalResult(self.name, query, out, (time.perf_counter()-t)*1000, kwargs)


## 2. Query analysis: classify the retrieval problem before acting

In [ ]:
BIOMEDICAL_RELATIONS = ["treats", "causes", "associated_with", "increases_risk", "decreases_risk", "interacts_with", "expressed_in", "measured_by", "prevents"]

def analyze_query(q: str) -> Dict[str, Any]:
    s=q.lower()
    qtype = "factoid"
    if any(x in s for x in ["why", "mechanism", "how does"]): qtype="mechanism"
    elif any(x in s for x in ["compare", "versus", "difference between"]): qtype="comparison"
    elif any(x in s for x in ["risk", "association", "associated"]): qtype="association"
    elif any(x in s for x in ["multiple", "which drugs", "what treatments", "list"]): qtype="list"
    elif any(x in s for x in ["effect", "affect", "impact"]): qtype="effect"
    entities = re.findall(r"\b[A-Z][A-Za-z0-9-]{2,}\b", q)
    relation_signal = any(r in s for r in BIOMEDICAL_RELATIONS) or any(w in s for w in ["relationship", "link", "pathway", "gene", "protein", "mutation"])
    multi_hop = relation_signal and (len(entities) >= 2 or any(w in s for w in ["through", "between", "pathway", "leads to"]))
    complexity = "high" if multi_hop or len(s.split()) > 28 or qtype in {"comparison","mechanism"} else ("medium" if len(s.split())>16 else "low")
    return {"question_type": qtype, "entities": entities, "relation_signal": relation_signal, "multi_hop_signal": multi_hop, "complexity": complexity}


## 3. Tool selection policy: bounded routing, not free-form planning

In [ ]:
TOOLS = ["bm25", "dense", "hybrid", "graph", "pageindex", "query2doc", "hyde", "decompose"]

def route_query(analysis: Dict[str, Any], budget_rounds=2) -> Dict[str, Any]:
    qt=analysis["question_type"]; comp=analysis["complexity"]
    primary=["hybrid"]
    recovery=[]
    rationale=["hybrid is the general biomedical default"]
    if analysis["multi_hop_signal"]:
        primary=["hybrid","graph"]
        rationale.append("multi-hop/relational signal -> graph candidate")
    if qt in {"mechanism","comparison"} or comp=="high":
        recovery += ["decompose", "pageindex"]
        rationale.append("complex reasoning -> decomposition/PageIndex recovery")
    if qt in {"factoid","association","effect"} and comp!="high":
        recovery += ["query2doc", "hyde"]
    # Keep the first round minimal; recovery tools are invoked only after insufficiency.
    return {"primary_tools": list(dict.fromkeys(primary)), "recovery_tools": list(dict.fromkeys(recovery)), "max_rounds": budget_rounds, "rationale": rationale}


## 4. Retrieval state machine

In [ ]:
@dataclass
class RetrievalState:
    question: str
    analysis: Dict[str, Any]
    plan: Dict[str, Any]
    round_id: int = 0
    history: List[Dict[str, Any]] = field(default_factory=list)
    evidence: List[Evidence] = field(default_factory=list)
    status: str = "NEW"

def dedupe_evidence(items):
    seen=set(); out=[]
    for x in items:
        if x.passage_id not in seen:
            seen.add(x.passage_id); out.append(x)
    return out

def evidence_assessment(question, evidence, gold_ids=None, min_k=3):
    n=len(evidence)
    if n==0: return {"status":"INSUFFICIENT","coverage_proxy":0.0,"redundancy":0.0}
    scores=np.array([e.score for e in evidence], dtype=float)
    coverage_proxy=float(np.clip(scores.mean() if len(scores) else 0,0,1))
    unique_sources=len(set(e.source for e in evidence))
    redundancy=1-unique_sources/max(1,n)
    gold_recall=None
    if gold_ids is not None:
        hit=len(set(gold_ids)&set(e.passage_id for e in evidence))
        gold_recall=hit/max(1,len(set(gold_ids)))
        coverage_proxy=max(coverage_proxy,gold_recall)
    status="SUFFICIENT" if n>=min_k and coverage_proxy>=0.45 else "WEAK"
    if gold_recall is not None and gold_recall==0: status="INSUFFICIENT"
    return {"status":status,"coverage_proxy":coverage_proxy,"redundancy":redundancy,"gold_recall":gold_recall}


## 5. Recovery operators: rewrite, HyDE, Query2Doc, decomposition

In [ ]:
def query2doc(q):
    return q + " relevant biomedical evidence mechanisms studies clinical findings"

def hyde(q):
    return "Hypothetical biomedical answer evidence: " + q

def decompose(q):
    parts=[]
    if " and " in q.lower():
        parts=[p.strip()+"" for p in re.split(r"\band\b", q, flags=re.I) if p.strip()]
    if not parts:
        parts=[q]
    return parts[:3]

def rewrite(q, failure="low_coverage"):
    return q + " " + ("key evidence" if failure=="low_coverage" else "biomedical mechanism")


## 6. Orchestrator implementation
A production version should inject the real adapters from Notebooks 07/08. The example below is executable with deterministic mock retrievers so the control logic can be unit-tested without pretending mock numbers are scientific results.

In [ ]:
def merge_results(results, top_k=10):
    all_e=[]
    for r in results: all_e.extend(r.evidence)
    all_e=dedupe_evidence(all_e)
    return sorted(all_e, key=lambda x:x.score, reverse=True)[:top_k]

def run_agent(question, adapters, gold_ids=None, top_k=10, budget_rounds=2):
    analysis=analyze_query(question); plan=route_query(analysis,budget_rounds)
    st=RetrievalState(question,analysis,plan,status="RUNNING")
    for rd in range(budget_rounds):
        st.round_id=rd+1
        tools=plan["primary_tools"] if rd==0 else plan["recovery_tools"]
        round_results=[]
        for tool in tools:
            adapter=adapters.get(tool)
            if adapter is None: continue
            qs=[question]
            if tool=="query2doc": qs=[query2doc(question)]
            elif tool=="hyde": qs=[hyde(question)]
            elif tool=="decompose": qs=decompose(question)
            elif tool=="bm25": qs=[rewrite(question)] if rd>0 else [question]
            for q in qs:
                round_results.append(adapter.retrieve(q,top_k=top_k))
        st.evidence=merge_results(round_results,top_k)
        assessment=evidence_assessment(question,st.evidence,gold_ids)
        st.history.append({"round":rd+1,"tools":tools,"assessment":assessment,"result_count":len(st.evidence)})
        if assessment["status"]=="SUFFICIENT":
            st.status="STOP"; break
    if st.status!="STOP": st.status="EXHAUSTED"
    return st


## 7. Deterministic smoke adapters

In [ ]:
def mock_fn(name):
    def fn(q, top_k=10, **kwargs):
        base=[]
        tokens=set(re.findall(r"[a-z0-9-]+", q.lower()))
        for i in range(min(top_k,8)):
            pid=f"mock-{name}-{i}"
            text=f"Synthetic {name} evidence for query: {q}"
            overlap=sum(t in text.lower() for t in tokens)/max(1,len(tokens))
            base.append(Evidence(pid,text,name,float(overlap)+0.05*(8-i),i+1))
        return base
    return fn

adapters={t:RetrieverAdapter(t,mock_fn(t)) for t in ["bm25","dense","hybrid","graph","pageindex","query2doc","hyde","decompose"]}

for q in [
    "What is the mechanism of action of aspirin in platelet aggregation?",
    "What proteins connect EGFR signaling to cell proliferation?",
    "What are the effects of metformin on type 2 diabetes?",
]:
    st=run_agent(q,adapters,budget_rounds=2)
    print("\nQ:",q,"\nanalysis:",st.analysis,"\nstatus:",st.status,"\nhistory:",st.history)


## 8. Fixed vs static multi-tool vs adaptive agentic baselines

In [ ]:
def fixed_hybrid(question, adapter, top_k=10):
    return adapter.retrieve(question,top_k=top_k)

def static_multi_tool(question, adapters, top_k=10):
    rs=[adapters[t].retrieve(question,top_k=top_k) for t in ["bm25","dense","graph","pageindex"] if t in adapters]
    return merge_results(rs,top_k)

def evaluate_single(q, adapters, gold_ids=None):
    t0=time.perf_counter(); fixed=fixed_hybrid(q,adapters["hybrid"]); fixed_ms=(time.perf_counter()-t0)*1000
    t0=time.perf_counter(); static=static_multi_tool(q,adapters); static_ms=(time.perf_counter()-t0)*1000
    t0=time.perf_counter(); agent=run_agent(q,adapters,gold_ids=gold_ids); agent_ms=(time.perf_counter()-t0)*1000
    def recall(e):
        if not gold_ids: return np.nan
        return len(set(gold_ids)&set(x.passage_id for x in e))/max(1,len(set(gold_ids)))
    return {"question":q,"fixed_recall":recall(fixed.evidence if hasattr(fixed,'evidence') else fixed),"static_recall":recall(static),"agent_recall":recall(agent.evidence),"fixed_ms":fixed_ms,"static_ms":static_ms,"agent_ms":agent_ms,"agent_rounds":agent.round_id}


## 9. Retrieval Decision Accuracy, Routing Regret, Help/Harm, and Agent Efficiency

In [ ]:
def summarize_agent_decisions(records: pd.DataFrame):
    out={}
    if len(records)==0: return out
    out["agent_mean_recall"]=float(records.agent_recall.mean()) if "agent_recall" in records else None
    out["fixed_mean_recall"]=float(records.fixed_recall.mean()) if "fixed_recall" in records else None
    out["static_mean_recall"]=float(records.static_recall.mean()) if "static_recall" in records else None
    out["mean_agent_rounds"]=float(records.agent_rounds.mean()) if "agent_rounds" in records else None
    out["agent_speedup_vs_static"]=float(records.static_ms.mean()/records.agent_ms.mean()) if records.agent_ms.mean()>0 else np.nan
    return out

def routing_regret(agent_score, oracle_score):
    return max(0.0,float(oracle_score)-float(agent_score))

def classify_help(no_retrieval_score, actual_score, oracle_score):
    if actual_score>no_retrieval_score: return "HELPED"
    if actual_score<no_retrieval_score: return "HURT"
    return "NEUTRAL"


## 10. Agent ablations
The critical experiments vary one control at a time: recovery enabled/disabled, max rounds, graph enabled/disabled, PageIndex enabled/disabled, expansion enabled/disabled, and sufficiency threshold.

In [ ]:
ABLATIONS=[
    {"name":"agent_full","max_rounds":2,"use_graph":True,"use_pageindex":True,"use_expansion":True},
    {"name":"no_recovery","max_rounds":1,"use_graph":True,"use_pageindex":True,"use_expansion":False},
    {"name":"no_graph","max_rounds":2,"use_graph":False,"use_pageindex":True,"use_expansion":True},
    {"name":"no_pageindex","max_rounds":2,"use_graph":True,"use_pageindex":False,"use_expansion":True},
    {"name":"no_expansion","max_rounds":2,"use_graph":True,"use_pageindex":True,"use_expansion":False},
]
print(pd.DataFrame(ABLATIONS))


## 11. Budget frontier
Agentic retrieval should be evaluated as a quality/compute frontier rather than as a single score. Measure Recall@K / answer quality against number of tool calls, latency, and estimated token cost.

In [ ]:
def efficiency_table(run_records):
    df=pd.DataFrame(run_records)
    if df.empty: return df
    for c in ["recall","quality","latency_ms","tool_calls","estimated_tokens"]:
        if c not in df: df[c]=np.nan
    df["quality_per_ms"]=df["quality"]/df["latency_ms"].clip(lower=1)
    df["quality_per_tool"]=df["quality"]/df["tool_calls"].clip(lower=1)
    return df


## 12. Failure taxonomy for agentic retrieval

In [ ]:
FAILURE_TAXONOMY={
"A01":"bad_query_analysis","A02":"wrong_tool_routing","A03":"unnecessary_tool_call","A04":"lexical_miss","A05":"dense_miss","A06":"graph_link_failure","A07":"graph_traversal_failure","A08":"pageindex_navigation_failure","A09":"query_expansion_drift","A10":"hyde_drift","A11":"decomposition_error","A12":"fusion_failure","A13":"reranker_failure","A14":"insufficient_evidence_not_recovered","A15":"over_retrieval","A16":"latency_budget_exceeded","A17":"conflicting_evidence","A18":"should_have_abstained"}
print(json.dumps(FAILURE_TAXONOMY,indent=2))


## 13. Production observability schema

In [ ]:
TRACE_FIELDS=[
"trace_id","question","question_type","complexity","plan","round","tool","query_variant",
"candidate_count","selected_count","retrieval_latency_ms","reranker_latency_ms","assessment",
"gold_recall","stop_reason","recovery_trigger","failure_code","model_version","index_version"
]
print("Trace fields:",len(TRACE_FIELDS))


## 14. Save artifacts

In [ ]:
manifest={
 "notebook":"09_agentic_retrieval",
 "title":"Bounded Agentic Retrieval",
 "comparisons":["fixed_hybrid","static_multi_tool","adaptive_agentic"],
 "max_rounds_default":2,
 "tools":TOOLS,
 "custom_metrics":["retrieval_decision_accuracy","routing_regret","recovery_success_rate","retrieval_help_rate","retrieval_harm_rate","agent_efficiency","unnecessary_retrieval_rate"],
 "failure_taxonomy":FAILURE_TAXONOMY,
 "scientific_note":"Mock adapters validate control logic only; publish benchmark results only with real BioRAG-X retrieval artifacts."
}
(ARTIFACT_DIR/"manifest.json").write_text(json.dumps(manifest,indent=2))
print((ARTIFACT_DIR/"manifest.json").read_text())


## 15. Interpretation checklist
Before claiming that agentic retrieval is better, report: (1) retrieval quality, (2) answer correctness/grounding, (3) latency p50/p95, (4) tool calls, (5) token/cost estimate, (6) recovery success and harm, (7) routing errors by question type, and (8) oracle gap. A gain that comes only from spending substantially more compute is not an automatic architectural win.

## Handoff to Notebook 10
Notebook 10 should consume the selected evidence set and implement **evidence selection → context reconstruction → structured generation → claim extraction → citation validation → regeneration/abstention**, followed by the Oracle / No-Retrieval / Actual generation experiments.